# Social Media Aware Cleaning: Spell-checking, Link removals, Abbreviations, and Emojis/Emoticons.

In [1]:
!pip install spark-nlp==6.3.3 pyspark==3.3.1
!pip install emoji
!pip install emoticon_fix

In [2]:
import re
import emoji
from emoticon_fix import emoticon_fix
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, regexp_extract, col
from pyspark.sql.types import StringType
from functools import reduce
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *

In [3]:
spark = sparknlp.start()

:: loading settings :: url = jar:file:/uufs/chpc.utah.edu/common/home/u1332544/my-python-venvs/spark-nlp/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /uufs/chpc.utah.edu/common/home/u1332544/.ivy2/cache
The jars for the packages stored in: /uufs/chpc.utah.edu/common/home/u1332544/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-88bbb2ba-f860-48f5-b926-ce465e17a64b;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in centr

26/04/25 22:20:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
training = spark.read.csv('../data/twitter_training.csv', header=True)
col_names = ["id", "topic", "sentiment", "content"]
training_df = training.toDF(*col_names)
training_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- content: string (nullable = true)



## Emoji Cleaning

### Emojis

In [5]:
def translate_emojis(text):
    if text is None:
        return None
    
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")
    text = text.replace(":", "")
    
    return text

In [6]:
emoji_udf = udf(translate_emojis, StringType())
training_df = training_df.withColumn("content", emoji_udf(training_df["content"]))
training_df.select("content").show(5, truncate=False)

[Stage 1:>                                                          (0 + 1) / 1]

+---------------------------------------------------------+
|content                                                  |
+---------------------------------------------------------+
|I am coming to the borders and I will kill you all,      |
|im getting on borderlands and i will kill you all,       |
|im coming on borderlands and i will murder you all,      |
|im getting on borderlands 2 and i will murder you me all,|
|im getting into borderlands and i can murder you all,    |
+---------------------------------------------------------+
only showing top 5 rows



### Emoticons

In [7]:
def translate_emoticons(text):
    if text is None:
        return None
    
    text = emoticon_fix(text)
    
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")
    text = text.replace(":", "")

    return text
    

In [8]:
emoji_emoticon_udf = udf(translate_emoticons, StringType())
training_df = training_df.withColumn("translate_emoticons", emoji_emoticon_udf(training_df["content"]))
training_df.select("content", "translate_emoticons").show(5, truncate=False)

+---------------------------------------------------------+----------------------------------------------------------+
|content                                                  |translate_emoticons                                       |
+---------------------------------------------------------+----------------------------------------------------------+
|I am coming to the borders and I will kill you all,      |I am coming to the borders and I will kill you all ,      |
|im getting on borderlands and i will kill you all,       |im getting on borderlands and i will kill you all ,       |
|im coming on borderlands and i will murder you all,      |im coming on borderlands and i will murder you all ,      |
|im getting on borderlands 2 and i will murder you me all,|im getting on borderlands 2 and i will murder you me all ,|
|im getting into borderlands and i can murder you all,    |im getting into borderlands and i can murder you all ,    |
+-----------------------------------------------

## Clean Up Website Links

In [9]:
# Open web domains
with open("tlds.json", "r") as file:
    data = json.load(file)

tlds = data["tlds"]

# Regex for finding possible website links
website_regex = r"(http\w+)|(thesun\.\w+)|(\b\w+(?:" + "|".join(map(re.escape, tlds)) + r")\w+\b)"

training_df = training_df.withColumn(
    "content",
    regexp_replace("content", website_regex, "")
)

training_df.select("content").show(5, truncate=False)

+---------------------------------------------------------+
|content                                                  |
+---------------------------------------------------------+
|I am coming to the borders and I will kill you all,      |
|im getting on borderlands and i will kill you all,       |
|im coming on borderlands and i will murder you all,      |
|im getting on borderlands 2 and i will murder you me all,|
|im getting into borderlands and i can murder you all,    |
+---------------------------------------------------------+
only showing top 5 rows



## Split Hashtags

In [10]:
def split_hashtags(text):
    if text is None:
        return None

    def split_tag(match):
        tag = match.group()[1:]
        tag = tag.replace("_", " ")
        tag = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
        return tag

    return re.sub(r'#\w+', split_tag, text)

In [11]:
hashtag_udf = udf(split_hashtags, StringType())
training_df = training_df.withColumn("hashtags_split", hashtag_udf(training_df["content"]))
training_df.select("content").show(5, truncate=False)

+---------------------------------------------------------+
|content                                                  |
+---------------------------------------------------------+
|I am coming to the borders and I will kill you all,      |
|im getting on borderlands and i will kill you all,       |
|im coming on borderlands and i will murder you all,      |
|im getting on borderlands 2 and i will murder you me all,|
|im getting into borderlands and i can murder you all,    |
+---------------------------------------------------------+
only showing top 5 rows



## Spell Check + Normalize

In [12]:
document_assembler = DocumentAssembler()\
  .setInputCol("content")\
  .setOutputCol("document")

tokenizer = RecursiveTokenizer()\
  .setInputCols(["document"])\
  .setOutputCol("token")\
  .setPrefixes(["\"", "(", "[", "\n"])\
  .setSuffixes([".", ",", "?", ")","!", "‘s"])

normalizer = Normalizer() \
    .setInputCols(["token"]) \
    .setOutputCol("normalized") \
    .setLowercase(False)\
    .setCleanupPatterns(["[^A-Za-z0-9]"])

spell_checker = NorvigSweetingModel.pretrained() \
    .setInputCols(["normalized"]) \
    .setOutputCol("spell_checked")

spellcheck_norvig download started this may take some time.
Approximate size to download 4.2 MB
[ | ]26/04/25 22:20:45 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/04/25 22:20:45 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
spellcheck_norvig download started this may take some time.
Approximate size to download 4.2 MB
Download done! Loading the resource.


[Stage 6:======================================>                   (8 + 4) / 12]

[ / ]

[OK!]


In [13]:
finisher = Finisher() \
          .setInputCols("spell_checked") \
          .setOutputCols(["cleaned_text"]) \
          .setOutputAsArray(False) \
          .setCleanAnnotations(False) \
          .setAnnotationSplitSymbol(" ")

spell_check_pipeline = Pipeline(stages = [document_assembler,
                                tokenizer,
                                normalizer,
                                spell_checker,
                                finisher])

In [14]:
training_df = spell_check_pipeline.fit(training_df).transform(training_df)
training_df.select("cleaned_text").show(5, truncate=False)

[Stage 7:>                                                          (0 + 1) / 1]

+--------------------------------------------------------+
|cleaned_text                                            |
+--------------------------------------------------------+
|I am coming to the borders and I will kill you all      |
|im getting on borderlands and i will kill you all       |
|im coming on borderlands and i will murder you all      |
|im getting on borderlands 2 and i will murder you me all|
|im getting into borderlands and i can murder you all    |
+--------------------------------------------------------+
only showing top 5 rows



## Remove abbreviations

In [15]:
with open("abbreviations.json", "r") as file:
    abbvs = json.load(file)

expr = reduce(
    lambda col, kv: regexp_replace(col, rf"(?i)\b{kv[0]}\b", kv[1]),
    abbvs.items(),
    col("cleaned_text")
)

training_df = training_df.withColumn("cleaned_text", expr)
training_df.select("cleaned_text").show(5, truncate=False)

+--------------------------------------------------------+
|cleaned_text                                            |
+--------------------------------------------------------+
|I am coming to the borders and I will kill you all      |
|im getting on borderlands and i will kill you all       |
|im coming on borderlands and i will murder you all      |
|im getting on borderlands 2 and i will murder you me all|
|im getting into borderlands and i can murder you all    |
+--------------------------------------------------------+
only showing top 5 rows



## Export CSV

In [16]:
cleaned_df = training_df.select(col("id"), 
                                col("topic"), 
                                col("sentiment").alias("og_sentiment"), 
                                col("cleaned_text"))

In [19]:
cleaned_df.toPandas().to_csv("../cleaned_data/social_media_cleaned_tweets.csv", index=False)